# 22 - Delivery
Core runs are complete. Run setup, then the cold-start cell and delivery cell. Registry, alert, and CI evidence cells can be skipped when their saved reports pass. Keep the WSL mlop-pj kernel.


In [1]:
import json
import os
import subprocess
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if p.name == "MLOP-PJ" and (p / "README.md").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
assert Path(sys.prefix).name == "mlop-pj", "Select the WSL mlop-pj kernel."
assert "microsoft" in os.uname().release.lower(), "Open this project in WSL."
status = json.loads((ROOT / "reports/azure-blob-upload.json").read_text())
assert status.get("data_uploaded") and status.get("all_hashes_match") and not status["errors"], "Finish 17 first."

def step(phase):
    print("Running:", phase, flush=True)
    process = subprocess.run([sys.executable, "-m", "cloudlayer.finish", phase],
                             cwd=ROOT, capture_output=True, text=True)
    if process.returncode:
        raise RuntimeError("Phase process failed. Check its stderr before retrying: " + process.stderr)
    result = json.loads(process.stdout)
    print(json.dumps(result, indent=2))
    if result["errors"]:
        raise RuntimeError(f"Check reports/finish-{phase}.json and the matching log in tmp. Stop this group.")
    if result.get("pending"):
        print("Still waiting. Read next in the report; do not recreate the resource or job.")
    return result

print("Root:", ROOT)
print("17 passed. Run this notebook work cells in order.")

Root: /mnt/c/Users/SN23341J001185/Documents/MyWork/ICT/ICTYear4_1/MLop/public_teaching_mlaiops/course/MLOP-PJ
17 passed. Run this notebook work cells in order.


## Group 5 — Final checks and delivery
After the training job and both GitHub runs finish, run this cell. It refreshes registry, alert, and CI evidence, writes the one-page model card, checks provider separation, and queries cost per 1,000 observed predictions.

`pending` means evidence is still missing. Billing can arrive later, so a planning estimate is kept separate from the reported cost. Read `docs/DELIVERY_STATUS.md` and `reports/finish-delivery.json` for the remaining items. Never replace pending results with a claimed pass.

If `gh` is missing, complete its setup in the guide before this cell. The code will stop on a real command error; rerun the relevant phase rather than Group 1.

In [2]:
registry = step("training-registry")
assert registry.get("registry_ready"), "Wait for registry verification before continuing."


Running: training-registry
{
  "checked_at": "2026-10-11T19:29:14.411267+00:00",
  "phase": "training-registry",
  "errors": [],
  "job_name": "train-20261011-164656",
  "status": "Completed",
  "registry_ready": true,
  "registered_name": "travel-style",
  "registered_version": "1",
  "fresh_fit_equivalent": true,
  "registration_reused": true
}


### Alert evidence
Refresh the real alert record. No new failure is injected.


In [3]:
alert = step("alert")


Running: alert
{
  "checked_at": "2026-10-11T19:33:50.349855+00:00",
  "phase": "alert",
  "errors": [],
  "alerts": [
    {
      "alertRule": "/subscriptions/b175b91e-058e-4a22-8e96-39e6e432b006/resourceGroups/rg-mlop-pj-492ea24d/providers/microsoft.insights/scheduledqueryrules/mlop-semantic-failure",
      "monitorCondition": "Resolved",
      "alertState": "New",
      "startDateTime": "2026-10-11T16:56:39.5744343Z",
      "lastModifiedDateTime": "2026-10-11T17:21:41.138289Z"
    }
  ],
  "alert_fired": true,
  "pending": false,
  "next": "Actual alert evidence found. Save the alert screenshot and continue to CI/CD."
}


### CI/CD evidence
Run after GitHub setup, the reviewed push, and both workflow runs. gh must be installed and logged in. If this fails, rerun only this cell after fixing GitHub; do not repeat registry.


In [4]:
ci_evidence = step("ci-evidence")


Running: ci-evidence
{
  "checked_at": "2026-10-11T19:33:58.542218+00:00",
  "phase": "ci-evidence",
  "errors": [],
  "workflow_published": true,
  "ci_success_run": "https://github.com/DewPerawis/MLOP-PJ/actions/runs/38166188586",
  "tests_blocked_deploy_runs": [
    "https://github.com/DewPerawis/MLOP-PJ/actions/runs/38166197980"
  ],
  "ci_evidence_ready": true,
  "pending": false
}


## Cold start
Close API tabs first. This checks zero replicas, then sends one readiness request. It may wait up to 10 minutes. Run delivery after `cold_start_measured: true`.


In [5]:
cold = step("cold")


Running: cold
{
  "checked_at": "2026-10-11T19:34:01.943397+00:00",
  "phase": "cold",
  "errors": [],
  "cold_start_measured": true,
  "zero_replicas_observed_at": "2026-10-11T19:34:10.519196+00:00",
  "revision": "travel-style-492ea24d--latest",
  "image_uri": "acrmloppj492ea24d.azurecr.io/travel-style@sha256:b55cd10aba3f8bb899f3cdec62cbc1815b5869ce22fc5676ebd06642382758a4",
  "url": "https://travel-style-492ea24d.icymushroom-08e3ded0.centralindia.azurecontainerapps.io",
  "first_request_ms": 3819.595852008206,
  "model_version": "abfd88e893ca",
  "scope": "WSL HTTPS readiness request after ARM reported zero replicas; includes network and scale activation."
}


### Delivery report
Run after the evidence cells. Read open_items and the separate latency/cold-start flags. A report file alone does not mean the project is ready to submit.


In [6]:
delivery = step("delivery")


Running: delivery
{
  "checked_at": "2026-10-11T19:34:15.091325+00:00",
  "phase": "delivery",
  "errors": [],
  "portability_passed": true,
  "checks": {
    "versioned_data_remote": true,
    "managed_training_registry": true,
    "deployed_inference": true,
    "ci_cd_evidence": true,
    "dashboard": true,
    "actual_alert_event": true,
    "failure_and_recovery": true,
    "controlled_input_drift_evidence": true,
    "cost_documented": true,
    "actual_cost_available": false,
    "model_card": true,
    "portability": true
  },
  "open_items": [
    "actual_cost_available"
  ],
  "required_evidence_complete": false,
  "latency_target_met": false,
  "cold_start_measured": true,
  "cleanup_verified": false
}


## Cleanup after saving evidence
Save the reports, workflow links, dashboard and alert screenshots, model card, training run, and registry lineage first. See `docs/RUN_FINISH.md` for the exact cleanup command. Cleanup deletes the whole project resource group, including uploaded data and models. Run it yourself only when those resources are no longer needed.

The reminder is 13 October 2026, 21:15 Bangkok time. Verify `reports/cleanup.json` after deletion. There is no automatic cleanup.